## Step 0: Install and Import Libraries

In [4]:
!pip install -q transformers datasets scikit-learn pandas openpyxl torch accelerate

In [3]:
import pandas as pd
import numpy as np
import torch

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder
from sklearn.metrics import accuracy_score, precision_recall_fscore_support, classification_report

from transformers import (
    AutoTokenizer,
    AutoModelForSequenceClassification,
    TrainingArguments,
    Trainer
)

from datasets import Dataset

SEED = 42
np.random.seed(SEED)
torch.manual_seed(SEED)

## Step 1: Load and Inspect the data

In [4]:
df = pd.read_excel("merged_text_dataset.xlsx")

print("Shape of the dataset:", df.shape)
print(df.head())
print("\nMissing values:\n", df.isnull().sum())
print("\nDuplicate rows:", df.duplicated().sum())
print("\nCategory distribution:\n", df["category"].value_counts())
print("\nUrgency level distribution:\n", df["urgency_level"].value_counts())

Shape of the dataset: (4344, 5)
                                         description        category  \
0  "घरायसी फोहोर ओभरफ्लो भएर व्यापारिक क्षेत्र को...  domestic_trash   
1  "Community drain partly blocked by leaves sinc...        drainage   
2  "Public bin overflowing onto the pavement, nea...           trash   
3  "Industrial drain filled with mud after rain a...        drainage   
4  "Cracked pavement creating a trip hazard at th...  infrastructure   

   urgency_score urgency_level location_type  
0            5.7        medium    commercial  
1            4.5        medium        public  
2            7.0          high       highway  
3            8.0          high        market  
4            3.5           low       highway  

Missing values:
 description      0
category         0
urgency_score    0
urgency_level    0
location_type    0
dtype: int64

Duplicate rows: 2

Category distribution:
 category
drainage             550
illegal_parking      548
road_damaged_sign    548

## Step 2: Clean and Preprocess the Data

In [8]:
def clean_text(text):
    text = str(text).strip()
    text = text.strip('"').strip("'")
    text = " ".join(text.split())
    return text

df["description"] = df["description"].apply(clean_text)

before = len(df)
df = df.drop_duplicates(subset=["description"]).reset_index(drop=True)
print(f"Dropped {before - len(df)} duplicate rows. Remaining: {len(df)}")

df["description"].sample(5, random_state=SEED).tolist()

Dropped 0 duplicate rows. Remaining: 4342


['The public bin is full and people are putting bags beside it. The smell is starting to bother people nearby.',
 'Road divider defaced with large drawings on a highway, sharp parts are exposed beside pedestrians.',
 'School boundary wall covered with spray-painted tags near an industrial road, damage is visible from the school gate.',
 'गाडीहरू अस्पताल को दुबै साइडमा गैरकानूनी पार्किङ गरेका छन्। चाँडै कारबाही लिनुपर्यो।',
 'There is a hole near the edge of the road where motorcycles pass. Vehicles are swerving to avoid the hole.']

## Step 3: Encode Labels

In [10]:
label_encoder = LabelEncoder()
df["category_label"] = label_encoder.fit_transform(df["category"])

num_labels = df["category_label"].nunique()
id2label = {i: label for i, label in enumerate(label_encoder.classes_)}
label2id = {label: i for i, label in id2label.items()}

print("Number of categories:", num_labels)
print("Label mapping:", id2label)

#Optional: same idea for urgency_label if we train a seperate urgency classifier
urgency_encoder = LabelEncoder()
df["urgency_label"] = urgency_encoder.fit_transform(df["urgency_level"])
urgency_id2label = {i: label for i, label in enumerate(urgency_encoder.classes_)}
print("Urgency label mapping:", urgency_id2label)

Number of categories: 9
Label mapping: {0: 'domestic_trash', 1: 'drainage', 2: 'garbage', 3: 'illegal_parking', 4: 'infrastructure', 5: 'pothole', 6: 'road_damaged_sign', 7: 'trash', 8: 'vandalism'}
Urgency label mapping: {0: 'critical', 1: 'high', 2: 'low', 3: 'medium'}


## Step 4: Split into train, validation, and tests sets

In [12]:
train_val_df, test_df = train_test_split(
    df, test_size=0.15, 
    stratify=df["category_label"], 
    random_state=SEED
)

train_df, val_df = train_test_split(
    train_val_df, test_size=0.1765,
    stratify=train_val_df["category_label"], random_state=SEED
)

print ("Train:", len(train_df), "Validation:", len(val_df), "Test:", len(test_df))
train_df.to_csv("train.csv", index=False)
val_df.to_csv("validation.csv", index=False)
test_df.to_csv("test.csv", index=False)

Train: 3038 Validation: 652 Test: 652


## Step 5: Tokenize with the MuRIL tokenizer

In [23]:
MODEL_NAME = "google/muril-base-cased"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

lengths = df["description"].apply(lambda t: len(tokenizer.tokenize(t)))
print("95th percentile token length:", int(np.percentile(lengths, 95)))
print("Max token length:", lengths.max())

MAX_LENGTH = 128 #adjust based on the printed percentile above
def tokenize_batch(examples):
    return tokenizer(
        examples["description"],
        padding="max_length",
        truncation=True,
        max_length=MAX_LENGTH,
    )

95th percentile token length: 25
Max token length: 33


In [24]:
train_ds = Dataset.from_pandas(train_df[["description", "category_label"]].rename(columns={"category_label": "label"}))
val_ds = Dataset.from_pandas(val_df[["description", "category_label"]].rename(columns={"category_label": "label"}))
test_ds = Dataset.from_pandas(test_df[["description", "category_label"]].rename(columns={"category_label": "label"}))

train_ds = train_ds.map(tokenize_batch, batched=True)
val_ds = val_ds.map(tokenize_batch, batched=True)
test_ds = test_ds.map(tokenize_batch, batched=True)

columns = ["input_ids", "attention_mask", "label"]
train_ds.set_format(type="torch", columns=columns)
val_ds.set_format(type="torch", columns=columns)
test_ds.set_format(type="torch", columns=columns)

Map:   0%|          | 0/3038 [00:00<?, ? examples/s]

Map:   0%|          | 0/652 [00:00<?, ? examples/s]

Map:   0%|          | 0/652 [00:00<?, ? examples/s]

## Step 6: Load and Configure the MuRIL Model

In [25]:
model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] BertForSequenceClassification LOAD REPORT from: google/muril-base-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params w